(dem_manager)=
# DEM Manager: Automatic Copernicus GLO-30 DEM

`DEMManager` turns a geographic extent into a ready-to-use Copernicus GLO-30 DEM without manual downloading: it queries the raw tiles that cover the extent, reuses tiles already present in a local cache, downloads the missing ones, and mosaics them into a single GeoTIFF.

This tutorial covers:

1. Listing the tiles that cover an area
2. Building a combined DEM from cached tiles
3. Choosing where the DEM is written
4. Downloading missing tiles on demand
5. Letting `run_pair` resolve the DEM automatically

> **Prerequisites.** This notebook assumes you are running inside the project's `.venv` with all dependencies installed. `DEMManager` needs a cache folder; if `FANINSAR_DEM_CACHE_DIR` is already set in your environment it is reused, otherwise a temporary folder is created.

## Imports and cache setup


In [1]:
import os
import tempfile
from pathlib import Path

from faninsar.processing.geometry import DEMManager, RasterDEM, get_dem_manager
from faninsar.query import BoundingBox

cache_dir = Path(os.environ.get("FANINSAR_DEM_CACHE_DIR") or tempfile.mkdtemp(prefix="dem_cache_"))
os.environ["FANINSAR_DEM_CACHE_DIR"] = str(cache_dir)
print("tile cache:", cache_dir)
manager = DEMManager(cache_dir)

tile cache: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles


## 1. Which tiles cover my area?

`required_tiles` returns the Copernicus GLO-30 tile identifiers that intersect a `BoundingBox` (or a plain `(min_lon, min_lat, max_lon, max_lat)` tuple in EPSG:4326). No cache files or network access are needed for this step.

In [2]:
roi = BoundingBox(99.5, 38.5, 100.5, 39.5)
tiles = manager.required_tiles(roi)
print(f"{len(tiles)} tile(s) cover the ROI")
for tile_dir, filename in tiles:
    print(tile_dir, filename)

4 tile(s) cover the ROI
N38_E099 Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif
N38_E100 Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif
N39_E099 Copernicus_DSM_COG_10_N39_00_E099_00_DEM.tif
N39_E100 Copernicus_DSM_COG_10_N39_00_E100_00_DEM.tif


## 2. Build a combined DEM

`fetch_dem(bounds, output_path)` ensures every required tile is available (cache hit, or download when missing) and mosaics them into one GeoTIFF. The mosaic is float32, single-band, EPSG:4326 with `NaN` nodata.

In [3]:
out_dir = Path(tempfile.mkdtemp(prefix="dem_out_"))
dem_path = manager.fetch_dem(roi, out_dir / "dem.tif")
print("written:", dem_path)

import rasterio

with rasterio.open(dem_path) as src:
    print("shape:", src.shape, "| crs:", src.crs, "| dtype:", src.dtypes[0])
    print("bounds:", src.bounds)

2026-08-03 20:14:19 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif


2026-08-03 20:14:19 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif


2026-08-03 20:14:19 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N39_00_E099_00_DEM.tif


2026-08-03 20:14:19 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N39_00_E100_00_DEM.tif


2026-08-03 20:14:28 | INFO | faninsar.processing.geometry.dem_manager | DEM mosaic written: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_out_d2jzuhkm/dem.tif shape=(1, 7200, 7200)


written: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_out_d2jzuhkm/dem.tif
shape: (7200, 7200) | crs: EPSG:4326 | dtype: float32
bounds: BoundingBox(left=98.99986111111112, bottom=38.00013888888889, right=100.99986111111112, top=40.00013888888889)


## 3. Default output location

When `output_path` is omitted, the mosaic lands in `<cache parent>/dem/<name>`. The name defaults to `dem.tif` and can be changed with the `FANINSAR_DEM_NAME` environment variable.

In [4]:
os.environ["FANINSAR_DEM_NAME"] = "merged.tif"
path = get_dem_manager().fetch_dem(roi)  # no output_path
print(path)
os.environ.pop("FANINSAR_DEM_NAME", None)

2026-08-03 20:14:28 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif


2026-08-03 20:14:28 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif


2026-08-03 20:14:28 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N39_00_E099_00_DEM.tif


2026-08-03 20:14:28 | INFO | faninsar.processing.geometry.dem_manager | DEM tile cache hit: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles/Copernicus_DSM_COG_10_N39_00_E100_00_DEM.tif


2026-08-03 20:14:36 | INFO | faninsar.processing.geometry.dem_manager | DEM mosaic written: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/dem/merged.tif shape=(1, 7200, 7200)


/Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/dem/merged.tif


'merged.tif'

## 4. Missing tiles are downloaded

With an empty cache, `fetch_dem` downloads the missing tiles from the public AWS S3 bucket (up to three attempts per tile) before mosaicking. The cell below uses a fresh cache; it needs network access to fetch the tile.

In [5]:
fresh = DEMManager(Path(tempfile.mkdtemp(prefix="dem_fresh_")))
try:
    fresh_path = fresh.fetch_dem(
        (94.5, 34.5, 95.5, 35.5),
        Path(tempfile.mkdtemp(prefix="dem_fresh_out_")) / "dem.tif",
    )
    print("downloaded and mosaicked:", fresh_path)
except Exception as exc:
    print("download skipped (no network?):", type(exc).__name__, exc)

2026-08-03 20:14:54 | INFO | faninsar.processing.geometry.dem_manager | DEM tile fetched: https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N34_00_E094_00_DEM/Copernicus_DSM_COG_10_N34_00_E094_00_DEM.tif (36111629 bytes)


2026-08-03 20:15:17 | INFO | faninsar.processing.geometry.dem_manager | DEM tile fetched: https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N34_00_E095_00_DEM/Copernicus_DSM_COG_10_N34_00_E095_00_DEM.tif (37264421 bytes)


2026-08-03 20:15:38 | INFO | faninsar.processing.geometry.dem_manager | DEM tile fetched: https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N35_00_E094_00_DEM/Copernicus_DSM_COG_10_N35_00_E094_00_DEM.tif (36113195 bytes)


2026-08-03 20:15:58 | INFO | faninsar.processing.geometry.dem_manager | DEM tile fetched: https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N35_00_E095_00_DEM/Copernicus_DSM_COG_10_N35_00_E095_00_DEM.tif (35803240 bytes)


2026-08-03 20:16:06 | INFO | faninsar.processing.geometry.dem_manager | DEM mosaic written: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_fresh_out_xaleud2l/dem.tif shape=(1, 7200, 7200)


downloaded and mosaicked: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_fresh_out_xaleud2l/dem.tif


## 5. Automatic DEM in the pipeline

With `FANINSAR_DEM_CACHE_DIR` set, `run_pair(..., dem=None)` resolves the DEM by itself: it determines the coverage from the ROI (or from the selected bursts when no ROI is given), builds the mosaic at `<output_dir>/dem/<name>`, and uses it for the pair. The environment-driven manager behind it is `get_dem_manager()`:

```python
from faninsar.processing.pipeline import run_pair

state = run_pair(
    reference,   # list of SAFE products (one or more frames)
    secondary,   # same frames for the secondary date
    output_dir="./pair",
    roi=BoundingBox(97.5, 38.7, 101.4, 40.0),  # optional
    dem=None,    # automatic DEM via the environment
)
```

The `faninsar frame` CLI offers the same flow: pass `--dem name.tif` with a bare file name to have it resolved (and built if missing) under `<output>/dem/`, or omit `--dem` entirely and let `run_pair` resolve it:

```bash
faninsar frame --reference ref.zip --secondary sec.zip \
    --output ./pair --roi 97.5,38.7,101.4,40.0 --dem my_dem.tif
```

In [6]:
# The exact manager the pipeline would use:
manager = get_dem_manager()
print("pipeline DEM manager cache:", manager.cache_dir)

pipeline DEM manager cache: /Volumes/DATA2/TEST_sentinel-1/faninsar-experiments/campaigns/PROPOSAL-0012/dem/tiles


## Exercise

Pick another ROI and answer:

1. How many tiles does it cover?
2. What is the first tile identifier?

```python
roi2 = BoundingBox(100.0, 39.0, 101.0, 40.0)
tiles2 = manager.required_tiles(roi2)
print("tiles:", len(tiles2))
print("first:", tiles2[0])
```

In [7]:
roi2 = BoundingBox(100.0, 39.0, 101.0, 40.0)
tiles2 = manager.required_tiles(roi2)
print("tiles:", len(tiles2))
print("first:", tiles2[0])

tiles: 4
first: ('N39_E100', 'Copernicus_DSM_COG_10_N39_00_E100_00_DEM.tif')


## Pitfalls and extensions

- **Bounds order.** All bounds are `(min_lon, min_lat, max_lon, max_lat)` in EPSG:4326; a swapped lat/lon order silently queries the wrong tiles.
- **Small tiles are ignored.** A cached tile smaller than 1 MiB is treated as missing and downloaded again.
- **Mirror sources.** Set `FANINSAR_DEM_SOURCE_URL` (or pass `source_url=` to `DEMManager`) to use a mirror of the default `https://copernicus-dem-30m.s3.amazonaws.com` bucket.
- **Sampling.** The written mosaic can be opened directly with `RasterDEM(dem_path, interpolation="biquintic")` and sampled at latitude/longitude coordinates.